# EDA: 데이터 분포 및 특성 분석

## 데이터셋 개요
- 출처: roboflow 공개 데이터셋
- 클래스: 원래 CLASS_NAMES 24개 중 카테고리별 대표 8개 선정
- 선정 기준: 귀중품/추억물품/가전/폐기물/서류/가구 각 카테고리 대표 클래스
- 총 이미지: train 922장, valid 220장, test 112장

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 라이브러리 import
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 경로 설정
# merged_cls: train/클래스명/이미지 구조
DATA_DIR = '/content/drive/MyDrive/해커톤/dataset/merged_cls/train'

# 카테고리 매핑 (cv_preprocessing.py CLASS_TO_CATEGORY 기반)
CLASS_TO_CATEGORY = {
    'TV':             '가전',
    'air_conditioning': '가전',
    'fridge':         '가전',
    'pet_bottle':     '폐기물',
    'chair':          '가구',
    'ring':           '귀중품',
    'paper_document': '서류',
    'album':          '추억물품',
}

class_names = list(CLASS_TO_CATEGORY.keys())
print(f'클래스 수: {len(class_names)}')
print(f'클래스 목록: {class_names}')

In [ ]:
# ① 클래스별 이미지 개수 분포 (불균형 확인)
class_counts = {}
for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        n = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        class_counts[cls] = n

df_counts = pd.DataFrame(list(class_counts.items()),
                          columns=['class', 'count']).sort_values('count', ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(data=df_counts, x='class', y='count', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=df_counts['count'].mean(), color='red', linestyle='--',
            label=f'Mean {df_counts["count"].mean():.0f}')
plt.title('Class-wise Image Count Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/eda_class_distribution.png', dpi=100)
plt.show()

print('Insight:')
print(f'  - 최대: {df_counts["count"].max()}장 ({df_counts.iloc[0]["class"]})')
print(f'  - 최소: {df_counts["count"].min()}장 ({df_counts.iloc[-1]["class"]})')
print(f'  - 불균형 비율: {df_counts["count"].max() / df_counts["count"].min():.1f}배')
print(f'  → album이 가장 적음. 소수 클래스 예측 성능 저하 위험 → 데이터 증강 필요')

In [ ]:
# ② 카테고리별 분포
df_counts['category'] = df_counts['class'].map(CLASS_TO_CATEGORY)
category_dist = df_counts.groupby('category')['count'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
category_dist.plot(kind='barh', color='steelblue')
plt.title('Category-wise Image Count')
plt.xlabel('Image Count')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/eda_category_distribution.png', dpi=100)
plt.show()

print('Insight:')
print(f'  - 가전 카테고리가 가장 많음 (TV + air_conditioning + fridge)')
print(f'  → 가전 편향으로 인해 다른 카테고리 예측 성능 저하 가능성')

In [ ]:
# ③ 이미지 크기/해상도 분포
sizes = []
for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    for fname in os.listdir(cls_path)[:30]:  # 클래스당 30장 샘플링
        try:
            img = Image.open(os.path.join(cls_path, fname))
            sizes.append({
                'class':        cls,
                'width':        img.width,
                'height':       img.height,
                'aspect_ratio': img.width / img.height
            })
        except:
            pass

df_sizes = pd.DataFrame(sizes)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(df_sizes['width'],        bins=30); axes[0].set_title('Width Distribution')
axes[1].hist(df_sizes['height'],       bins=30); axes[1].set_title('Height Distribution')
axes[2].hist(df_sizes['aspect_ratio'], bins=30); axes[2].set_title('Aspect Ratio')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/eda_image_size.png', dpi=100)
plt.show()

print('Insight:')
print(f'  - 이미지 크기가 다양함 → 224×224 리사이즈 시 정보 손실 가능성')
print(f'  - 작은 이미지는 늘리고, 큰 이미지는 줄여서 왜곡 발생 가능')

In [ ]:
# ④ 클래스별 평균 RGB 분석
def get_mean_rgb(img_path):
    img = np.array(Image.open(img_path).convert('RGB').resize((100, 100)))
    return img.reshape(-1, 3).mean(axis=0)

rgb_data = []
for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:20]
    rgbs = [get_mean_rgb(os.path.join(cls_path, f)) for f in files]
    mean_rgb = np.mean(rgbs, axis=0)
    rgb_data.append({'class': cls, 'R': mean_rgb[0], 'G': mean_rgb[1], 'B': mean_rgb[2]})

df_rgb = pd.DataFrame(rgb_data)
df_rgb.set_index('class')[['R', 'G', 'B']].plot(
    kind='bar', figsize=(12, 5), color=['red', 'green', 'blue'], alpha=0.7
)
plt.title('Mean RGB Value per Class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/eda_rgb.png', dpi=100)
plt.show()

print('Insight:')
print('  - ring(반지)은 R값이 높음(금속 반사광)')
print('  - TV/fridge는 전체적으로 밝은 색상')
print('  - album/paper_document는 색상이 유사 → 모델 혼동 가능성')

In [ ]:
# ⑤ 클래스별 샘플 이미지 미리보기
n_classes = len(class_counts)
cols = 4
rows = (n_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, cls in enumerate(sorted(class_counts.keys())):
    cls_path = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not files:
        continue
    sample = files[0]
    img = Image.open(os.path.join(cls_path, sample))
    axes[i].imshow(img)
    axes[i].set_title(f'{cls}\n({class_counts[cls]}장)')
    axes[i].axis('off')

# 남은 axes 숨기기
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/eda_sample_images.png', dpi=100)
plt.show()

print('Insight:')
print('  - album은 이미지 수가 가장 적어 학습 불리')
print('  - ring, pet_bottle은 배경이 다양해 학습 어려움 예상')

## EDA 종합 인사이트

1. **클래스 불균형**: album(43장) vs chair/fridge(200장+) → 데이터 증강 필수
2. **카테고리 편향**: 가전 카테고리 비중이 높아 가전 위주로 예측될 위험
3. **이미지 크기 다양**: 224×224 리사이즈 시 정보 손실 발생 가능
4. **색상 유사성**: album과 paper_document는 색상이 유사해 모델 혼동 예상
5. **데이터 한계**: roboflow 공개 데이터셋으로 실제 한국 유품과 차이 있을 수 있음